# KV caching: reuse an unchanged causal prefix

Chapter 4 · Session 3. We replay six fixed feature vectors through two residual attention layers. This is a small computation-graph example, not a pretrained Transformer and not free text generation. It deliberately omits norms, MLPs, and positional encodings. The output head has five arbitrary classes.

The goal is equality: incremental cached computation should match recomputing each full prefix.

In [ ]:
from pathlib import Path
import sys
import math
import torch
import torch.nn.functional as F
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/dongxi_llms/attention_evidence.py').exists())
if str(root / 'src') not in sys.path: sys.path.insert(0, str(root / 'src'))
from dongxi_llms.causal_attention_lab import attention_trace, teaching_inputs
from dongxi_llms.attention_evidence import gradient_evidence, scaling_evidence, cache_fixture, full_stack, decode_step, cache_evidence
torch.set_printoptions(precision=5, sci_mode=False)
print('PyTorch', torch.__version__, '| CPU float64')

## 1. Prediction: what is the identity of a cache entry?

If the input vector at the second position is identical in two sequences but the first position changes, must that second position have identical keys at every layer?

In [ ]:
prediction_identity = ""

### Reference solution

In this example, the first-layer projection is positionwise, so the identical second input has an identical first-layer key. After one causal attention update it can depend on the changed earlier position; its second-layer key may differ. This clarifies why context matters without claiming that every first-layer Q/K/V already contains contextual information.

In [ ]:
x, layers, head = cache_fixture()
changed = x.clone()
changed[0] += 3
_, original = full_stack(x[:2], layers, head)
_, altered = full_stack(changed[:2], layers, head)
for layer in range(len(layers)):
    error = (original[layer][0][1] - altered[layer][0][1]).abs().max().item()
    print('Layer', layer+1, '| second-position key difference:', error)
print('Identical second input:', torch.equal(x[1], changed[1]))

## 2. Implementation: prefill and one new query

Prefill two positions with `full_stack`. The result includes one K/V pair per layer. For the next position calculate q, append its k/v, and use q to read all available keys. Why do past queries not appear in this step?

The query is at the last global position. All provided cache positions are allowed. A naive 1×N lower-triangular mask would incorrectly keep only the first key.

In [ ]:
def my_last_query(q, old_k, old_v, new_k, new_v):
    all_k = ...
    all_v = ...
    weights = ...
    return weights @ all_v
run_my_step = False

### Reference solution

The new query determines today's retrieval. Past keys provide addresses and past values supply messages. Old queries have already completed their retrievals. Reuse applies to each layer's incoming states, so caches must stay separate across layers.

In [ ]:
def reference_last_query(q, old_k, old_v, new_k, new_v):
    all_k = torch.cat([old_k, new_k], dim=0)
    all_v = torch.cat([old_v, new_v], dim=0)
    weights = (q @ all_k.T / math.sqrt(q.shape[-1])).softmax(-1)
    return weights @ all_v

_, caches = full_stack(x[:2], layers, head)
wq, wk, wv = layers[0]
arguments = (x[2:3] @ wq, *caches[0], x[2:3] @ wk, x[2:3] @ wv)
print('First-layer retrieval:', reference_last_query(*arguments))
if run_my_step:
    torch.testing.assert_close(my_last_query(*arguments), reference_last_query(*arguments))
actual, updated = decode_step(x[2:3], caches, layers, head)
expected, _ = full_stack(x[:3], layers, head)
print('Cached logits:', actual)
print('Full-prefix logits:', expected[-1:])
torch.testing.assert_close(actual, expected[-1:], atol=1e-12, rtol=1e-12)

## 3. Prediction and verification: append repeatedly

Should numerical agreement hold only for the first new token, or for every extension? State the assumptions: fixed parameters, correct layer-specific state, compatible positions, unchanged prefix, no dropout, and compatible numerical execution.

In [ ]:
prediction_equivalence = ""

### Reference solution

The argument repeats at each layer and each new position. Earlier causal states are unchanged, their stored projections remain valid, and each new row can be computed using them. Small rounding differences are possible between execution strategies; we test at a declared tolerance.

In [ ]:
_, caches = full_stack(x[:2], layers, head)
for t in range(2, len(x)):
    actual, caches = decode_step(x[t:t+1], caches, layers, head)
    expected, _ = full_stack(x[:t+1], layers, head)
    difference = (actual - expected[-1:]).abs().max().item()
    print('Position', t, '| max error:', difference, '| K shapes:', [tuple(k.shape) for k, _ in caches])
    torch.testing.assert_close(actual, expected[-1:], atol=1e-12, rtol=1e-12)

## 4. Break it: reuse the wrong prefix

Reuse the old two-position cache after changing the first input. The next input row is unchanged. Can the runtime repair the prefix by processing just that new row?

In [ ]:
prediction_stale = ""

### Reference solution

No. The old keys/values represent the original prefix and propagate its state. Recompute the changed prefix (or its affected suffix with a valid earlier cache). Token identity alone does not authorize cache reuse.

In [ ]:
_, stale = full_stack(x[:2], layers, head)
stale_logits, _ = decode_step(changed[2:3], stale, layers, head)
correct, _ = full_stack(changed[:3], layers, head)
error = (stale_logits - correct[-1:]).abs().max().item()
print('Stale-cache logit error:', error)
assert error > 1e-6

## 5. Accounting: what was saved, and what grows?

Predict which grows as positions accumulate: parameter count, stored K/V bytes, or the number of keys the next query reads?

In [ ]:
prediction_cost = ""

### Reference solution

The model's parameter count is unchanged. Logical K/V storage and next-query attention work grow with the retained sequence. For this equal-width single-head fixture, bytes=2×layers×positions×width×bytes_per_scalar. This counts tensor payload, not allocator-reserved memory.

With a two-position prefill followed by four one-position appends, caching projects 12 layer-position rows. Recomputing prefixes of lengths 2,3,4,5,6 projects 40. These are work counts for K or V projections; they are not measured speedup factors.

In [ ]:
result = cache_evidence()
print(result)
logical_bytes = sum(k.numel()*k.element_size() + v.numel()*v.element_size() for k, v in caches)
print('Final logical cache bytes:', logical_bytes)
assert logical_bytes == 2 * 2 * 6 * 4 * 8 == 768

## 6. Lifecycle and evidence boundary

What happens when generation ends? Does releasing the cache necessarily reduce memory reported by the GPU allocator?

In [ ]:
my_cache_boundary = ""

### Reference solution

A completed request no longer needs its cache. Removing the runtime's references permits tensor storage to be freed or reused, but an allocator may retain reserved memory. Some serving systems retain compatible exact prefixes for reuse in later requests. A later conversation turn needs retained state or prefix recomputation.

Our replay verifies two-layer mathematical reuse under fixed execution. It does not benchmark serving frameworks, prove cache reuse under parameter changes, or test positional encodings, beams, eviction, or quantization. See the [chapter](../../book/chapters/04-attention-and-the-causal-information-boundary.md) and [worked solutions](../../book/solutions/04-attention-and-the-causal-information-boundary.md).